# tensor .item() — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-item-scalar`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `.item()` patterns that ramp from 0-D extract → single-elem extract → dtype preservation → `.item()` vs `.tolist()` → tensor → Python control flow. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `tensor-item-scalar`**, which bridges to the bank subtopic `Numpy: Core array literacy` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-item-scalar"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## `.item()` — quick refresher

**What it does.** Extracts a Python scalar (float / int / bool) from a tensor that has exactly one element. Bridges tensor space to Python space.

**Requirements.** `x.numel() == 1`. Shape doesn't matter — `(1,)`, `(1, 1)`, `()` all work as long as there's exactly one element. Multi-element tensors raise `RuntimeError`.

**Dtype mapping:**
- `float32` / `float64` → Python `float`
- `int32` / `int64` → Python `int`
- `bool` → Python `bool`

**When to use:** logging losses, control-flow conditions, returning counts to callers, dict / set keys (tensors aren't hashable).

**When NOT to use:** inside tight inner loops on GPU — every `.item()` is a host-device sync that blocks the kernel queue.

**For more elements:** `.tolist()` returns a (nested) Python list of any shape.

### Exercise 1 — extract a Python float from a 0-D tensor

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall how to extract a Python scalar from a 0-D tensor.
> Keywords: item, 0-D-tensor, python-float
> ```

**KCs targeted:** `item-from-zero-dim`

Implement `ex1_scalar_to_float(x)` to return the Python `float` value of a 0-D float tensor.

Use `x.item()`. The return type must be a plain Python `float`, NOT a tensor.

In [ ]:
def ex1_scalar_to_float(x: Tensor) -> float:
    """Extract a Python float from a 0-D float tensor."""
    raise NotImplementedError()


def _test_ex1():
    x = t.tensor(3.5)
    out = ex1_scalar_to_float(x)
    assert isinstance(out, float), f'expected float, got {type(out).__name__}'
    assert out == 3.5, f'expected 3.5, got {out}'
    # Negative.
    assert ex1_scalar_to_float(t.tensor(-2.0)) == -2.0
    # Note: 0-D from a reduction.
    assert ex1_scalar_to_float(t.tensor([1.0, 2.0, 3.0]).sum()) == 6.0
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_scalar_to_float(x: Tensor) -> float:
    return x.item()
```

**Why `.item()` and not `float(x)`?** Both work for 0-D float tensors. But `.item()` is the official PyTorch API, gives a clear error on wrong-shape input, and works uniformly across float / int / bool. `float(x)` is older Python coercion machinery — it works here but doesn't generalise. Default to `.item()`.
</details>

### Exercise 2 — extract from a single-element 1-D tensor

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply `.item()` to a single-element tensor of arbitrary shape.
> Keywords: 1-elem, any-shape, scalar-tensor
> ```

**KCs targeted:** `item-from-one-elem-1d`

Implement `ex2_one_elem_extract(x)`. `x` is a tensor containing exactly one element, but it might have any shape — `(1,)`, `(1, 1)`, `(1, 1, 1)`. Return the Python scalar value.

`.item()` works on any single-element tensor, no matter how many size-1 axes wrap it.

In [ ]:
def ex2_one_elem_extract(x: Tensor) -> float:
    """Extract the Python scalar from a single-element tensor."""
    raise NotImplementedError()


def _test_ex2():
    assert ex2_one_elem_extract(t.tensor([7.0])) == 7.0
    assert ex2_one_elem_extract(t.tensor([[7.0]])) == 7.0
    assert ex2_one_elem_extract(t.tensor([[[7.0]]])) == 7.0
    # Result must be a Python float, not a tensor.
    out = ex2_one_elem_extract(t.tensor([[2.5]]))
    assert isinstance(out, float), f'expected float, got {type(out).__name__}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_one_elem_extract(x: Tensor) -> float:
    return x.item()
```

**Why `.item()` doesn't care about wrapper axes.** Internally PyTorch checks `x.numel() == 1`, not `x.dim() == 0`. So any tensor with exactly one element — regardless of how many `(1, 1, 1, …)` wrappers — works. This is handy when you've kept axes around via `keepdim=True` and the reduction collapsed to a single scalar.
</details>

### Exercise 3 — .item() preserves dtype family

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `.item()` across dtypes and recognize that the Python return type follows the tensor's dtype family.
> Keywords: dtype-mapping, int-vs-float, bool
> ```

**KCs targeted:** `item-dtype-preservation`

Implement `ex3_describe_scalar(x)`. Given a single-element tensor of any dtype, return a tuple `(value, python_type_name)` where:
- `value = x.item()`
- `python_type_name` is the result of `type(value).__name__` — e.g. `'int'`, `'float'`, `'bool'`.

Float dtypes (`float32`, `float64`) produce Python `float`. Integer dtypes (`int64`, `int32`) produce Python `int`. `bool` produces `bool`.

In [ ]:
def ex3_describe_scalar(x: Tensor) -> tuple:
    """Return (x.item(), type name string)."""
    raise NotImplementedError()


def _test_ex3():
    v, name = ex3_describe_scalar(t.tensor(3.5))
    assert v == 3.5 and name == 'float', f'float32: got ({v!r}, {name!r})'
    v, name = ex3_describe_scalar(t.tensor(7, dtype=t.long))
    assert v == 7 and name == 'int', f'long: got ({v!r}, {name!r})'
    v, name = ex3_describe_scalar(t.tensor(True))
    assert v is True and name == 'bool', f'bool: got ({v!r}, {name!r})'
    # double precision still maps to Python float.
    v, name = ex3_describe_scalar(t.tensor(2.0, dtype=t.float64))
    assert v == 2.0 and name == 'float', f'float64: got ({v!r}, {name!r})'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_describe_scalar(x: Tensor) -> tuple:
    v = x.item()
    return v, type(v).__name__
```

**Practical implication.** If you `.item()` an `int64` tensor and pass it to a function that expects `float`, you'll silently get integer division behavior in Python 2-style edge cases (today rare, but the `/` vs `//` distinction still matters). Always check what dtype you started with before consuming the scalar.

**Numerical precision.** `float32` tensors lose precision when `.item()`'d to Python `float` (which is 64-bit) — but the *value* was already truncated in the tensor. `.item()` doesn't help or hurt.
</details>

### Exercise 4 — .item() vs .tolist() — pick the right tool

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `.item()` for single-element tensors and `.tolist()` for multi-element ones — and detect when the wrong one would error.
> Keywords: tolist, shape-sensitive, error-handling
> ```

**KCs targeted:** `item-vs-tolist-vs-many`

Implement `ex4_extract_safely(x)`. Given a tensor of unknown size, return:
- A Python scalar (via `.item()`) if `x` has exactly one element.
- A nested Python list (via `.tolist()`) otherwise.

Use `x.numel()` to check the size. Don't blindly call `.item()` — it raises `RuntimeError` on multi-element tensors.

In [ ]:
def ex4_extract_safely(x: Tensor):
    """Scalar for 1-elem tensors, nested list otherwise."""
    raise NotImplementedError()


def _test_ex4():
    # Scalar case.
    out = ex4_extract_safely(t.tensor([42.0]))
    assert isinstance(out, float) and out == 42.0, f'1-elem should give float, got {out!r}'
    # Multi-element 1-D.
    out = ex4_extract_safely(t.tensor([1.0, 2.0, 3.0]))
    assert out == [1.0, 2.0, 3.0], f'1-D list mismatch: {out!r}'
    # 2-D → nested list.
    out = ex4_extract_safely(t.tensor([[1.0, 2.0], [3.0, 4.0]]))
    assert out == [[1.0, 2.0], [3.0, 4.0]], f'2-D list mismatch: {out!r}'
    # 0-D — single element.
    assert ex4_extract_safely(t.tensor(5.0)) == 5.0
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_extract_safely(x: Tensor):
    if x.numel() == 1:
        return x.item()
    return x.tolist()
```

**The error you avoid.** `t.tensor([1.0, 2.0]).item()` raises `RuntimeError: a Tensor with 2 elements cannot be converted to Scalar`. Branching on `numel()` keeps your utility code working across shapes.

**When to use `.tolist()`.** For dumping a small tensor to JSON, logging structured data, or hand-comparing values during debugging. For large tensors it's wasteful — prefer summary stats.
</details>

### Exercise 5 — tensor → Python control flow

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize tensor reduction + `.item()` to bridge tensor space to Python space, returning a Python-typed result usable in `if` / `range` / logging.
> Keywords: control-flow, reduce-then-item, logging-loop, multi-kc
> ```

**KCs targeted:** `item-from-zero-dim`, `item-dtype-preservation`, `item-for-python-control-flow`

Implement `ex5_count_above(x, threshold)`. The canonical 'tensor → Python scalar for control flow' pattern.

Given a 2-D tensor `x` of shape `(B, D)` and a scalar `threshold`, count how many rows have L2 norm strictly greater than `threshold`. Return a plain Python `int` (NOT a 0-D tensor).

Steps:
1. Compute per-row L2 norms — shape `(B,)`.
2. Build a bool mask `norms > threshold`.
3. Sum the mask to get a 0-D `int64` tensor.
4. `.item()` to get a Python `int`.

Why a Python int? So callers can use the count in `if count > 0:` without `.item()` boilerplate, or pass it to `range(...)`.

> ⚠️ **Integrative exercise.** Combines 3 KCs (0-D-extract, dtype preservation, tensor→Python control flow bridge). Empirical work (Lohr et al. ITiCSE 2025) shows 3-concept exercises drop to ~40% solvability — expect a step up vs Exercises 1-4.

In [ ]:
def ex5_count_above(x: Tensor, threshold: float) -> int:
    """Count rows with L2 norm > threshold, return a Python int."""
    raise NotImplementedError()


def _test_ex5():
    x = t.tensor([
        [3.0, 4.0],   # norm 5
        [0.0, 0.0],   # norm 0
        [1.0, 0.0],   # norm 1
        [6.0, 8.0],   # norm 10
    ])
    # threshold = 0.5: rows 0, 2, 3 qualify (norms 5, 1, 10).
    count = ex5_count_above(x, threshold=0.5)
    assert isinstance(count, int), f'expected int, got {type(count).__name__}'
    assert count == 3, f'expected 3, got {count}'

    # threshold = 4.0: rows 0, 3 qualify (norms 5, 10).
    assert ex5_count_above(x, threshold=4.0) == 2

    # threshold = 100.0: no rows qualify.
    zero_count = ex5_count_above(x, threshold=100.0)
    assert isinstance(zero_count, int) and zero_count == 0

    # Must be a Python int — usable in range().
    consumed = list(range(ex5_count_above(x, threshold=0.5)))
    assert consumed == [0, 1, 2], 'result must be usable in range()'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_count_above(x: Tensor, threshold: float) -> int:
    norms = x.pow(2).sum(dim=1).sqrt()
    mask = norms > threshold
    return mask.sum().item()
```

**Why a Python int and not a 0-D tensor?** Callers that use the result as a loop bound, condition, or array length need a Python scalar. PyTorch is fine with most operator overloads, but `range(t.tensor(5))` raises in some versions, and a 0-D tensor stored in a dict key won't hash. `.item()` is the unambiguous bridge.

**Cost.** `.item()` is a host-device sync if `x` is on GPU — every call blocks the kernel queue. Inside a tight training loop you should minimise `.item()` calls (cache the loss tensor, `.item()` only when logging) but for one-off control flow it's fine.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()